# 세 모델 답 비교하기 — 원본 · 내 모델 · GPT
같은 시험 문제(`test.jsonl`, 학습에 안 쓴 5문제)를 세 모델에 물어보고 나란히 비교해요.

| 순서 | 칸 | 만들어지는 것 |
|---|---|---|
| 1 | 환경 준비 · 설정 | — |
| 2 | 세 모델 답 받기 | `answers.jsonl` (한 번 받으면 저장돼서 다시 묻지 않아요) |
| 3 | 표로 보기 · 자동 체크 요약 | 화면에 표 |
| 4 | 웹 비교표 | `compare_report.html` (브라우저로 열림) |

준비: `test.jsonl`(1_split_data에서 생성), Ollama에 원본 `qwen2.5:3b`와 내 모델 등록, `.env`에 `OPENAI_API_KEY`

In [1]:
# ▶ 1. 환경 준비 · 설정 — 여기만 고치세요
import os, re, json, html, random
from datetime import datetime
from pathlib import Path
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

MY_MODEL = 'mbcsports'          # ollama list 에 나온 내 모델 이름
BLIND    = False                # True: 이름을 가리고 A·B·C로 섞기 (학생 블라인드 평가용)
RERUN    = False                # True: 저장된 답을 무시하고 세 모델에 다시 묻기

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
test = pd.read_json('test.jsonl', lines=True)
local = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')   # 내 PC의 Ollama
cloud = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

MODELS = {'원본 Qwen2.5-3B': (local, 'qwen2.5:3b'),
          '내가 학습한 모델': (local, MY_MODEL),
          'gpt-4.1-mini':    (cloud, 'gpt-4.1-mini')}
print('시험 문제', len(test), '개 · API 키', '준비됨' if OPENAI_API_KEY else '없음')

시험 문제 5 개 · API 키 준비됨


In [5]:
# ▶ 2. 세 모델 답 받기 — 약 1~2분. 결과는 answers.jsonl 에 저장
ANS_FILE = Path('answers.jsonl')

def ask(client, model, row):
    msg = row['instruction'] + '\n\n' + row['input']
    try:
        r = client.chat.completions.create(model=model, messages=[{'role': 'user', 'content': msg}], temperature=0)
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f'(실패: {type(e).__name__})'

if ANS_FILE.exists() and not RERUN:
    answers = json.loads(ANS_FILE.read_text(encoding='utf-8'))
    print('저장된 답을 불러왔어요 (다시 물으려면 RERUN = True)')
else:
    answers = {}
    for name, (c, m) in MODELS.items():
        print('묻는 중:', name)
        answers[name] = [ask(c, m, r) if c else '(키 없음)' for _, r in test.iterrows()]
    ANS_FILE.write_text(json.dumps(answers, ensure_ascii=False, indent=1), encoding='utf-8')
    print('✅ 저장:', ANS_FILE)

묻는 중: 원본 Qwen2.5-3B
묻는 중: 내가 학습한 모델
묻는 중: gpt-4.1-mini
✅ 저장: answers.jsonl


In [6]:
# ▶ 3-1. 표로 보기 — 글 전체를 다 보여 줘요
pd.set_option('display.max_colwidth', None)
view = pd.DataFrame({'정답 리드': test['output'], **answers})
view.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap', 'vertical-align': 'top'})

,정답 리드,원본 Qwen2.5-3B,내가 학습한 모델,gpt-4.1-mini
0,다음 달 중동 원정에 나서는 축구대표팀 명단이 발표됐습니다. 모레노 감독은 손흥민과 은퇴 소동을 벌인 96년생 3인방 선수들을 모두 발탁해 정면 돌파 의지를 내비쳤는데요. 주장은 손흥민이라고 못을 박았습니다. 박재웅 기자입니다.,"MBC 뉴스 박재웅입니다. 오늘은 축구대표팀의 중요한 결정이 발표되었습니다. 역대 최장수 캡틴 손흥민의 리더십에 불만을 털어놓으며 은퇴 소동을 벌였던 김민재와 황인범, 황희찬. 모레노 임시 감독은 다음 달 중동 원정 2연전을 앞두고 손흥민과 96년생 동기 3인방을 모두 발탁했습니다. 이들은 경기력 측면에서, 조국을 위해 뛰겠다는 의지 측면에서도 최고의 선수로 평가받고 있습니다. 모레노 감독은 내부 문제는 대화를 통해 해결하는 게 원칙이라면서도, 선수들과 직접 대화를 나눈 내용은 공개하지 않겠다고 밝혔습니다. 주장 교체 가능성에 대해서는 단호히 선을 그었고, 김민재와 황인범의 경기를 각각 관전하겠다는 계획을 밝혔습니다. 이에 따라, 모레노 감독의 면담 내용에 대한 관심이 높아지고 있습니다. 오늘의 뉴스, MBC 뉴스 박재웅입니다.","축구대표팀의 내부 문제를 일으킨 김민재와 황인범, 황희찬이 손흥민과 함께 주장직을 담당하는 가운데 모레노 감독이 임시 감독 체제에서 치르는 마지막 평가전에선 96년생 선수들을 중심으로 팀을 이끌겠다고 밝혔습니다.","역대 최장수 주장 손흥민과 ‘96년생 3인방’ 김민재, 황인범, 황희찬이 불화설을 딛고 다시 한 팀으로 뭉쳤습니다. 다음 달 중동 원정 2연전을 앞두고 로베르트 모레노 임시 감독이 이들을 모두 발탁하며 정면 돌파에 나섰는데요. 갈등의 중심에 선 선수들과 감독의 대화 내용, 그리고 대표팀의 향방에 관심이 쏠리고 있습니다. 자세한 소식, 박재웅 기자가 전해드립니다."
1,"프로야구 한국 시리즈에서 삼성이 kt를 꺾고 통산 9번째 우승을 달성했습니다. 정규시즌 2위로 열세가 예상됐지만 1패 뒤 4연승으로 짜릿한 역전 우승을 이루며, 12년 만에 정상에 복귀했는데요. 김태운 기자가 보도합니다.","MBC 뉴스 김태운입니다. 1차전 패배 후 3연승으로 사실상 9부 능선을 넘은 삼성 라이온즈. 이 상승세는 멈출 줄을 몰랐습니다. 한 점 뒤진 상황에서, 구자욱의 적시타로 동점을 만든 뒤, 디아즈의 역전 적시타로 분위기는 단숨에 휘어집니다. 하지만, 다급해진 KT는 선발 고영표를 투입해도 삼성 타선을 막지 못했습니다. 5회부터 시작된 삼성의 공세는 김영웅과 이재현의 연속 적시타로 더욱 가속페달을 밟았고, 이어진 만루 상황에서 강민호의 실책으로 5회에만 8점을 더 불러들였습니다. 결국, 9회 마지막 아웃카운트를 잡아내며, 삼성은 12년 만에 통산 9번째 우승을 확정했습니다. 박진만 감독은 이 우승이 선수들의 노력 덕분이라며 감격했습니다. 이 경기는 라이온즈파크에서 열린 10년 만의 시즌 우승으로 마무리됩니다. 감격적인 순간들이 이어졌습니다.","삼성의 9부 우승에 나선하면서, 삼성 라이온즈는 12년 만에 정규시즌 우승을 확정했습니다. 김태운 기자입니다.","삼성이 1차전 패배의 아쉬움을 딛고 3연승을 달리며 사실상 우승의 9부 능선을 넘었습니다. 치열한 승부 끝에 12년 만에 통산 9번째 한국시리즈 정상에 오른 삼성, 홈팬들의 뜨거운 환호 속에 박진만 감독은 부임 4년 만에 감독으로서 첫 우승을 일궈냈습니다. 자세한 소식, 김태운 기자가 전해드립니다."
2,여자 기계체조 평균대에서 금메달을 따낸 '작은 거인' 황서현 선수가 하루 사이에 말 그대로 깜짝 스타로 떠올랐습니다. 오늘 입국장에서도 가장 큰 주목을 받았던 황서현의 금메달 뒷이야기를 일본 나고야 현지에서 곽승규 기자가 전합니다.,"나고야에서 MBC 뉴스 곽승규입니다. 135cm의 작은 체구에도 너비 10cm의 평균대 위에서 강단에 서는 황서현. 그녀의 연기는 강단에 서는 순간부터 강렬했습니다. 하지만 40년 만의 아시안게임 금메달이란 소식은 그녀를 놀라게 했습니다. ""금메달은 상상 못했어요. 금메달은 상상 못했고, 진짜 잘하면 동메달까지는 생각해 봤어요."" 황서현 선수의 진심 어린 말은 그녀의 열정과 노력의 결과를 보여줍니다. 새로운 스타의 탄생은 황서현의 금메달 소식을 전한 유튜브 뉴스 영상으로도 폭발적인 반응을 얻었습니다. 그녀의 우승 순간, 태극기를 들고 무대에 오르려다 제지당하는 장면까지 화제가 되었고, 황서현 선수는 자신의 SNS에 직접 ""머쓱했다""는 말을 남기기도 했습니다. 그녀의 금메달은 10년간의 힘든 훈련과 고난을 통해 이루어졌습니다. 7살 때부터 시작한 기계체조는 그녀에게 큰 재능을 인정받았지만, 그 과정은 고된 훈련과 지치는 날들이었습니다. 그러나 황서현의 엄마는 ""작지만 누구보다 큰 아이""라는 말로 그녀의 성공을 축하하며, 그녀의 꿈을 이룰 수 있도록 응원했습니다. 황서현 선수는 귀국 후 다음 달 전국체전과 세계선수권 대회에 출전할 예정이며, 그녀의 학교생활과 함께 새로운 도약을 준비하고 있습니다. 금메달의 꿈을 이룬 황서현의 이야기는 앞으로의 길을 더욱 밝게 만들 것입니다. 나고야에서 MBC 뉴스 곽승규입니다.","여자기계체조 황서현 선수의 금메달이 뜨거운 관심을 받고 있습니다. 135cm의 작은 체구에도 강한 실력으로 40년 만에 아시안게임 금메달을 차지했는데요. 황서현 선수는 자신의 SNS에 ""금메달은 상상 못했고, 진짜 잘하면 동메달까지는 생각해 봤어요""라고 밝혔습니다.","아시안게임 40년 만에 여자 기계체조 금메달이 탄생했습니다. 키 135cm의 작은 체구에도 불구하고 평균대 위에서 당당한 연기를 펼친 황서현 선수, 누구도 예상하지 못한 쾌거를 이뤄냈습니다. 오늘 황서현 선수의 감동적인 이야기, 자세히 전해드립니다."
3,축구대표팀이 우루과이에 완패한 데 이어 베네수엘라와도 졸전 끝에 득점 없이 비겼는데요. 김민재 선수가 축구화를 집어던지며 화를 낸 뒤 경기 후 작심발언을 쏟아냈습니다. 박재웅 기자입니다.,"MBC 뉴스 박재웅입니다. 최근 우루과이전에서 완패를 면치 못하고, 피파 랭킹 48위를 기록하며 베네수엘라와도 득점 없이 무승부를 기록한 축구대표팀. 김민재 선수의 불만이 이어지면서, 공격수들의 전방 압박과 수비 가담 부족이 문제라는 주장이 제기되며, 김민재 선수는 경기 후 쓴소리를 쏟아냈습니다. 황인범 선수는 협회장 부재와 모레노 감독의 임시감독 지위 등 현실적인 어려움을 호소하며, 김민재 선수의 불만이 축구대표팀의 문제를 더욱 부각시키고 있습니다. 현재 모레노 감독이 맡은 6경기 중 절반인 3경기가 지났음에도, 선수들의 불만이 계속적으로 터져나오고 있습니다. 이에 대해선 어떻게든 해결책을 찾아야 할 시점입니다. MBC 뉴스 박재웅입니다.","축구대표팀이 모레노 감독의 임시 감독단을 맡은 가운데, 선수들의 불만이 계속 터지고 있습니다. 김민재 선수가 후반 35분 교체될 때까지 절친 황인범과도 불화설이 일고 있는 가운데, 모레노호는 시작부터 흔들리고 있습니다. 박재웅 기자입니다.","한국 축구 대표팀이 우루과이에 이어 베네수엘라와의 평가전에서도 답답한 경기 끝에 득점 없이 비겼습니다. 선수들 사이에서는 전술보다 팀워크 문제를 지적하는 목소리가 나오고, 주장 손흥민과 김민재의 미묘한 입장 차도 드러났습니다. 모레노 감독 체제 출발부터 흔들리는 대표팀, 현장의 분위기 전해드립니다."
4,배드민턴 안세영 선수가 49연승 행진 속에 프랑스오픈 결승에 올랐는데요. 대회 3연패와 함께 시즌 10번째 우승을 눈앞에 두게 됐습니다.,"안녕하세요, MBC 뉴스입니다. 오늘은 스포츠 리포트를 전해드리겠습니다. 준결승에서 세계 9위 일본의 오쿠하라를 상대로 첫 게임에서 21대 12로 완승을 거두며, 두 번째 게임에서도 7대 3까지 끌려갔지만, 안세영은 7연속 득점을 통해 경기를 뒤집고 주도권을 되찾았습니다. 하지만, 앵커님, 안세영의 매서운 대각선 공격에 연거푸 몸을 날려야 했던 오쿠하라는 지친 모습이었습니다. 결국, 안세영은 2대 0으로 결승 진출을 확정지었습니다. 올 시즌 49연승 행진을 이어가며, 안세영은 65승 1패의 경이적인 승률을 과시했습니다. 경쟁자 야마구치와 왕즈이가 일찌감치 탈락하면서, 대

In [7]:
# ▶ 3-2. 자동 체크 — 리드 형식을 지켰나? (참고용, 최종 판단은 사람이)
def checks(ans, src, gold):
    a = ans.replace('\\n', '\n')
    sentences = [s for s in re.split(r'(?<=[.!?다요])\s+', a.strip()) if s.strip()]
    extra = sorted(n for n in set(re.findall(r'\d[\d,.]*', a)) if n.strip('.,') not in src)
    ratio = len(a) / max(len(gold), 1)
    greet = bool(re.search(r'MBC\s*뉴스\s*\S{2,4}입니다|안녕하세요', a))
    return {'끝인사 없음': not greet, '2문장 이하': len(sentences) <= 2,
            '길이 비슷(0.5~2배)': 0.5 <= ratio <= 2, '본문에 없는 숫자 없음': not extra,
            '_문장수': len(sentences), '_배율': ratio, '_숫자': extra, '_인사': greet}

def failed(a): return (not a) or a.startswith('(실패') or a.startswith('(키')

rows = []
for name, outs in answers.items():
    res = [checks(a, r['input'], r['output']) for a, (_, r) in zip(outs, test.iterrows()) if not failed(a)]
    n = len(res)
    rows.append({'모델': name, '답한 문제': f'{n}/{len(outs)}',
                 **{k: f'{sum(x[k] for x in res)}/{n}' for k in ['끝인사 없음', '2문장 이하', '길이 비슷(0.5~2배)', '본문에 없는 숫자 없음']},
                 '평균 길이(정답 대비)': f"{sum(x['_배율'] for x in res)/max(n,1):.1f}배"})
summary = pd.DataFrame(rows).set_index('모델')
summary

,답한 문제,끝인사 없음,2문장 이하,길이 비슷(0.5~2배),본문에 없는 숫자 없음,평균 길이(정답 대비)
모델,,,,,,
원본 Qwen2.5-3B,5/5,0/5,0/5,0/5,5/5,4.3배
내가 학습한 모델,5/5,5/5,3/5,4/5,5/5,1.0배
gpt-4.1-mini,5/5,5/5,0/5,4/5,5/5,1.6배


In [8]:
# ▶ 4. 웹 비교표 만들기 → compare_report.html (브라우저로 바로 열려요)
REPORT = Path('compare_report.html')
COLORS = {'원본 Qwen2.5-3B': '#9aa5b1', '내가 학습한 모델': '#1f8a70', 'gpt-4.1-mini': '#2563a8'}

def badges(c):
    out = [('기자 끝인사·인사말 있음' if c['_인사'] else '끝인사 없음', c['끝인사 없음']),
           (f"{c['_문장수']}문장", c['2문장 이하']),
           ('길이 비슷' if c['길이 비슷(0.5~2배)'] else f"길이 {c['_배율']:.1f}배", c['길이 비슷(0.5~2배)']),
           ('본문에 없는 숫자 없음' if not c['_숫자'] else '본문에 없는 숫자: ' + ', '.join(c['_숫자'][:3]), c['본문에 없는 숫자 없음'])]
    return ''.join(f'<span class="b {"ok" if ok else "no"}">{html.escape(t)}</span>' for t, ok in out)

def card(head, color, ans, r):
    if failed(ans):
        body = f'<div class="ans empty">{html.escape(ans or "답 없음")}</div>'
    else:
        body = (f'<div class="ans">{html.escape(ans.replace(chr(92)+"n", chr(10)))}</div>'
                f'<div class="meta">{badges(checks(ans, r["input"], r["output"]))}</div>')
    return (f'<div class="card"><div class="h" style="background:{color}">{html.escape(head)}</div>{body}'
            '<label class="human"><input type="checkbox"> 사실 오류 있음</label>'
            '<label class="human"><input type="checkbox"> 가장 좋은 답</label></div>')

names = list(answers)
blocks, key_rows = [], []
for i, (_, r) in enumerate(test.iterrows()):
    order = names[:]
    if BLIND: random.shuffle(order)
    heads = ['ABC'[k] if BLIND else n for k, n in enumerate(order)]
    key_rows.append(f'<tr><td>문제 {i+1}</td>' + ''.join(f'<td>{"ABC"[k]} = {html.escape(n)}</td>' for k, n in enumerate(order)) + '</tr>')
    cards = ''.join(card(h, '#18232f' if BLIND else COLORS.get(n, '#18232f'), answers[n][i], r) for h, n in zip(heads, order))
    blocks.append(
        f'<h2>문제 {i+1}</h2>'
        f'<details><summary>리포트 본문 보기 ({len(r["input"])}자)</summary><pre class="q">{html.escape(r["input"])}</pre></details>'
        f'<div class="gold"><b>정답 리드</b> {html.escape(r["output"])}</div><div class="row">{cards}</div>')

sum_html = summary.reset_index().to_html(index=False, classes='sum', border=0)
key_html = ('<details class="key"><summary>🔑 정답지 열기 (다 고른 뒤에)</summary><table class="k">'
            + ''.join(key_rows) + '</table></details>') if BLIND else ''
mode = '블라인드 모드: 문제마다 A·B·C 순서를 새로 섞었어요. 같은 글자라도 문제마다 다른 모델이에요.' if BLIND \
       else '순서 고정: 원본(회색) → 내가 학습한 모델(초록) → gpt-4.1-mini(파랑)'

page = f'''<!doctype html><html lang="ko"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>세 모델 비교</title><style>
body{{font-family:"Malgun Gothic",sans-serif;margin:24px auto;max-width:1500px;background:#f7f7f5;color:#222;padding:0 16px}}
h1{{font-size:26px}} h2{{font-size:20px;margin-top:40px;border-top:2px solid #ddd;padding-top:20px}}
.mode{{background:#fff;border:1px solid #ddd;border-radius:8px;padding:10px 14px}}
.sum{{border-collapse:collapse;background:#fff;margin:10px 0}} .sum th,.sum td{{border:1px solid #ddd;padding:6px 12px;text-align:center}}
.sum th{{background:#18232f;color:#fff}}
.q{{background:#fff;border:1px solid #ddd;padding:12px;white-space:pre-wrap;border-radius:8px;line-height:1.6}}
.gold{{background:#fff7ed;border-left:6px solid #e8590c;padding:10px 14px;margin:10px 0;line-height:1.7}}
.row{{display:flex;gap:12px;align-items:stretch}}
.card{{flex:1;background:#fff;border:1px solid #ddd;border-radius:10px;overflow:hidden;display:flex;flex-direction:column}}
.h{{color:#fff;font-weight:bold;padding:8px 12px;font-size:17px}}
.ans{{white-space:pre-wrap;line-height:1.7;padding:12px;flex:1}} .empty{{color:#aaa;text-align:center}}
.meta{{padding:0 12px 8px;font-size:12px}}
.b{{display:inline-block;padding:2px 8px;border-radius:10px;margin:2px 4px 2px 0}}
.ok{{background:#d8f3e8;color:#16664c}} .no{{background:#f6dede;color:#8a2a2a}}
.human{{display:block;font-size:13px;color:#555;padding:4px 12px;border-top:1px dashed #eee}}
details summary{{cursor:pointer;color:#2563a8;margin:6px 0}}
.key{{margin-top:40px;background:#fff;border:2px dashed #999;padding:12px;border-radius:8px}} .k td{{padding:4px 10px}}
</style></head><body>
<h1>같은 문제, 세 모델의 리드 비교</h1>
<p>만든 시각: {datetime.now():%Y-%m-%d %H:%M} · 초록 배지 = 리드 형식을 지킴, 빨강 = 확인 필요 (자동 확인은 참고용)</p>
<p class="mode">{mode}</p>
<h2 style="border:none;margin-top:16px">자동 체크 요약</h2>{sum_html}
<p>자동 체크는 <b>형식</b>만 봐요. <b>사실이 맞는지</b>는 리포트 본문을 펼쳐서 사람이 직접 대조하고, 카드 아래 체크박스에 표시하세요.</p>
{''.join(blocks)}
{key_html}
</body></html>'''
REPORT.write_text(page, encoding='utf-8')
print('✅ 비교표 완성:', REPORT.resolve())
if os.name == 'nt':
    os.startfile(REPORT.resolve())

✅ 비교표 완성: C:\workspace\ai-datajournalism-labs\4_파인튜닝\compare_report.html


## 사람이 확인할 것 — 자동 체크가 못 보는 부분
- **사실이 본문과 맞나?** 본문에 없는 내용(원인·전망·인용 출처)을 지어내지 않았나
- **누가 한 말인가?** 인터뷰 발언을 SNS 글이라고 쓰는 식의 바꿔치기
- **말이 자연스러운가?** "나선하면서", "선수생"처럼 어색한 표현

⚠ `answers.jsonl`과 `compare_report.html`에는 기사 내용이 들어 있어요. GitHub에 올리지 않아요.